# ARGUS-FEDER — Answering GA's disruption competency questions

GA's competency-question set (27 questions) splits into an ELM half and a
disruption half. This notebook works through the **disruption half, Q14–Q27**,
against the stored disruption index built from `disruption-py` output.

Two things to keep in mind throughout: GA's questions are written for **three**
disruption pipelines and we have run **one**; and the index holds **20 shots,
balanced 10 disrupted / 10 not**, so it is a demonstration slice rather than a
sample of the archive. Several questions below are answered correctly by
declining to answer.

Every question below is typed in plain English into a `%%ask` cell. The agent
decides what to look up, writes and runs its own code, and reports back.
**Nothing here requires you to write code.**

Questions are marked `--review`: before an answer reaches you, a grader
sub-agent checks it against a stated rubric and sends it back for revision if
it fails. Watch for the verdict panel under each answer.

#### Step 1 — Prepare your session (one-time)

Run the cell below. **Your session will restart automatically — that is
expected.** Once it comes back, carry on with Step 2.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

#### Step 2 — Choose your AI model

Pick one and run the cell (defaults to a free NRP model).

In [ ]:
# Use GLM-5 from National Research Platform
# LLM = {
#     "model": "glm-5",
#     "url": "https://ellm.nrp-nautilus.io/v1",
#     "api_key_env": "NRP_API_KEY",
# }

# Use GPT-5.5 from OpenAI
# LLM = {
#     "model": "gpt-5.5",
#     "api_key_env": "OPENAI_API_KEY",
# }

# Use Claude Sonnet 4.6 from Anthropic
# LLM = {
#     "model": "claude-sonnet-4-6",
#     "api_key_env": "ANTHROPIC_API_KEY",
#     "flavor": "anthropic",
# }

# Use Gemini 3.5 Flash from Google
# LLM = {
#     "model": "gemini-3.5-flash",
#     "api_key_env": "GOOGLE_API_KEY",
#     "flavor": "gemini",
# }

# Use GLM-5 from ZAI
LLM = {
    "model": "glm-5",
    "url": "https://api.z.ai/api/coding/paas/v4",
    "api_key_env": "ZAI_API_KEY",
}

#### Step 3 — Add your keys

Click the key icon in the left sidebar and add your secrets (toggle
"Notebook access" on for each):

- The API key matching what you picked in Step 2 (e.g. `NRP_API_KEY`)
- **Optional:** your DIII-D Pelican access token, named `FDP_TOKEN`. Every
  question in this notebook is answered from the stored index and needs no
  raw-signal access.

#### Step 4 — Install ARGUS-FEDER

In [ ]:
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/klinucsd/argus_feder/main/install.py'
).read().decode(), globals())

In [ ]:
%reset

#### Step 5 — Upload the disruption index

The questions read from one SQLite file, which is not bundled with the
installer. Open the **file browser** (folder icon, left sidebar) and drag it
in; it lands in `/content/`, where the skill looks automatically.

| file | size | needed for |
|---|---|---|
| `disruption.sqlite` | ~2 MB | every question |

The upload takes seconds, but **must be repeated for each fresh Colab
session** — Colab discards local files when the runtime is recycled.

Do not mount Google Drive for this: uploading one small file per session is
faster than the mount flow and does not hand the notebook access to your whole
Drive.

#### Step 6 — Confirm the index was found

Plain Python, no agent. Prints the file it located, its provenance, and the
coverage every answer below is conditional on.

In [ ]:
import os, sys
sys.path.insert(0, os.path.expanduser("~/.deepagents/agent/skills/d3d-disruption"))
from d3d_disruption import index_info, locate_disruption_db

print("database:", locate_disruption_db())
i = index_info()
print(f"{i['n_shots']} shots ({i['n_disrupted']} disrupted), {i['n_sample_rows']:,} sample rows")
print(f"produced by {i['package']} {i['version']} @ {i['commit_sha'][:8]} on {i['produced_at'][:10]}")
print()
print(i['coverage_note'])

## Q20 — Which pipeline produced a label, at what version, from which inputs?

*provenance · **answerable***

Our assessment predicted this would become "✅ by construction" once we ran a
pipeline ourselves and recorded what we ran.

In [ ]:
%%ask --review
For the disruption labels in the index: which code produced them, at what
version and commit, when was it run and by whom, and what were its inputs? Show
me the full provenance record.

## Q15 — Which quantities are disruption precursors, and what measures each?

*precursor map · **answerable***

The assessment noted that `disruption-py`'s feature set *is* the precursor
map — n=1 mode amplitude, betap, li, q95, radiated fraction, locked mode,
v_loop. Now that it has been run, that map is queryable with units attached.

In [ ]:
%%ask --review
Which quantities in this dataset are disruption precursors, and what does
each one measure? Group them by the physics method that produces them, and give
the units for each.

## Q19 — For a given shot, what is the label and the time, and do the indicators agree?

*label agreement · **partially answerable***

GA asks whether three pipelines agree. We have one — but that one carries **two
independent disruption indicators**, and they disagree on some shots.

In [ ]:
%%ask --review
For DIII-D shot 194128, give me the disruption label and the disruption
time. Then tell me whether the dataset contains more than one disruption
indicator, whether they agree, and on which shots they disagree.

---
# B. Answerable for one pipeline only

## Q14 — What definition of "disruption" does the pipeline implement?

*definition · **one of three pipelines***

In [ ]:
%%ask --review
What definition of a disruption, and of the disruption time, does the
pipeline behind these labels use? Where is that definition documented, and how
would I check it? Be clear about which pipeline this is and which ones are
missing.

## Q22 — Which shots have every input the pipeline requires?

*input availability · **partially answerable***

In [ ]:
%%ask --review
Which parameters does this pipeline produce, and how completely are they
populated across the shots in the index? Identify any parameters that are
missing for a substantial fraction of rows, and say which shots are affected.

## Q25 — Can a hazard-model dataset be assembled?

*survival analysis · **partially answerable***

A hazard model needs, per shot, the time it entered flattop and the time it
disrupted or was censored. The second is available; the first is the open
question.

In [ ]:
%%ask --review
I want to build a hazard model of disruptions, which needs for each shot
the time it entered the flattop phase and the time it disrupted or ended
without disrupting. Can this dataset provide both? Give me whichever parts it
can and state plainly what is missing.

---
# C. Correctly refused — and the refusal is the finding

## Q17 — What fraction of a campaign disrupted?

*rate · **correctly refused***

Two independent blockers: the index is a constructed slice rather than a
sample, and GA's question excludes *intentional* disruptions, which nothing in
the data flags.

In [ ]:
%%ask --review
What fraction of DIII-D shots in the 2023–2024 campaigns disrupted,
excluding intentional disruptions?

## Q18 — Which discharge phase was each disruption in?

*phase · **blocked, and the reason is instructive***

The dataset contains a column intended to answer exactly this. It does not
work — and how the agent handles that is the point of the question.

In [ ]:
%%ask --review
For the disrupted shots in the index, which phase of the discharge was
each disruption in — ramp-up, flattop, or ramp-down?

---
# D. What would unblock the rest

## Summary

*synthesis · **answerable***

In [ ]:
%%ask --review
Across everything in this notebook: which of GA's disruption competency
questions can this index answer today, which need something more, and what
specifically is missing in each case? Be concrete about what would have to
arrive for each blocked question to become answerable.